# MSUCA - Reproducibility Notebook

_Samuel Abdul, Arkesh Das, Zachary Kozlowski, Justin Wijaya_

Welcome to the reproducibility notebook! The goal of this notebook is to ensure that you have the proper data files needed to generate the Cirriculum Map Degree Plans using MSU Registrar and CNS Majors data. 

## Table of Contents

1. [Regenerating the data files](#1-regenerating-the-data-files)  
2. [Ensuring your environment is set up correctly](#2-ensuring-your-environment-is-set-up-correctly)  
3. [Loading in a degree plan](#3-loading-in-a-degree-plan)  
4. [Visualizing a degree plan](#4-visualizing-a-degree-plan)  
5. [Getting metrics on a degree plan](#5-getting-metrics-on-a-degree-plan)


## 1. Regenerating the data files

### 1.1 Complete the install instructions

If you have not done so, please follow steps 1-11 in [install.md](../install.md).


### 1.2 Ensure that the outputs directory is empty.

Before you run any commands to regenerate the data, you must ensure that the outputs directory is empty. This is mostly a precautionary step, and if you just pulled the repo the outputs directory should be empty by default.

To do this, run the following command from the repository's root directory:

```bash
rm outputs/*.csv 
```


### 1.3 Getting CSV files from the Resgistrar and Majors data

Now that the outputs folder is clear, we can now generate the CSV files that the Cirricular Analytics package will use to generate the degree plan visualizations.

Assuming that you followed the installation steps, you should be able to run the following command to regenerate the output CSV files. 

```bash
uv run python python/scripts/build_ca_curricula.py \
  --registrar "data/20250919_Registrars_Data(in).csv" \
  --majors "data/CNS_Majors_Data.xlsx" \
  --output-dir outputs
```
What this is doing is that it's running the `build_ca_curricula.py` with the specified `registrar` and `majors` datasets. 

NOTE: If you are using NEW majors and registrar data, ensure that the file names match the file names of the files listed in the command above!

If this command worked properly, you should see the following text or something similar outputed in your terminal window:

```bash
Loading input files...
Trying registrar CSV encoding: utf-8
Loaded registrar rows: 20,101, major rows: 4,393, plans: 102, registrar courses with extracted prereqs: 3,590
Generating CA curriculum CSV files...
Actuarial Science BS (MTH 103): 19 courses with prereqs, 3 courses with coreqs
Actuarial Science BS (MTH 116): 19 courses with prereqs, 1 courses with coreqs
Actuarial Science BS (MTH 132): 19 courses with prereqs, 0 courses with coreqs
Advanced Mathematics BA from Calculus II: 2 courses with prereqs, 2 courses with coreqs
Advanced Mathematics BA from Linear Algebra: 2 courses with prereqs, 2 courses with coreqs
Processed 50 of 102 plans...
Processed 100 of 102 plans...
Done. Wrote 102 curriculum files to 'outputs'.
Summary written to: outputs/_summary.csv
```


## 2. Ensuring your environment is set up correctly

### 2.1 Test if base Julia is working

This notebook should be using a Julia-based Jupyter kernel, so if you are able to run the following code cell, your kernel is set-up correctly.

In [ ]:
1+1

### 2.2 Test if the main packages are working
If the cell above ran, then that means your Julia kernel is set up properly. Now, ensure that the kernel contains the proper packages. If the following cell is able to run, then the Julia kernel contains the required packages neeed to generate the degree plan visualizations.

using CurricularAnalytics
using CurricularVisualization
using CSV
using DataFrames

## 3. Loading in a degree plan

To load in a degree plan, we'll need to load in the regenerated data file. For this notebook, we'll use the Actuarial Science BS degree plan for students who need to take MTH 103:

In [ ]:
curric = read_csv("../outputs/Actuarial_Science_BS_MTH_103.csv")

## 4. Visualizing a degree plan

Now that the degree plan has been loaded into the curric variable, you can generate an interactive visualization by calling visualize(curric). This will start a local web server and open the curriculum graph in your browser, where you can explore each course, its credit hours, and its prerequisite relationships visually. Each node in the graph represents a course, and the directed edges between nodes represent prerequisite dependencies, meaning an edge from Course A to Course B indicates that Course A must be completed before Course B can be taken. The size or color of each node may reflect metrics such as blocking factor or complexity, giving you an a sense of which courses are the most structurally significant in the degree plan.

## 5. Getting metrics on a degree plan

### There are 4 main attributes that the curricular analytics package computes
Some of them use more advanced math, but here is a simple breakdown, along with some limitations:

- **Blocking Factor (Take this early):**  
  This measures how many courses depend on a given class. A course with a high blocking factor is a prerequisite for many others, so taking it early is important to avoid holding up progress. This is generally a reliable metric because it directly reflects how much a course “unlocks” the rest of the curriculum.

- **Delay Factor (Don’t postpone this):**  
  This measures how much a course can delay graduation if it is taken late. Courses with a high delay factor are part of long prerequisite chains, so pushing them back can extend your timeline. Like blocking factor, this is useful for identifying courses that are critical to staying on track.

- **Centrality (Important connector):**  
  This measures how connected a course is within the overall curriculum network. A course with high centrality sits in the middle of many prerequisite paths *(has many courses connecting to it in the graph)*. However, this metric can be misleading. Just because a course is highly connected does not necessarily mean it is the most important or difficult course. As the team explored different degrees, some highly central courses are not especially challenging or foundational they simply connect different parts of the curriculum. Meanwhile, some truly important courses may have lower centrality but are essential for understanding future material. **Because of this, centrality should be interpreted with caution.**

- **Complexity (Overall difficulty):**  
  This measures how complicated the entire curriculum is based on the number and arrangement of prerequisite relationships. Higher complexity means the curriculum is more rigid and harder to navigate. However, complexity reflects structure rather than actual course difficulty, so it does not always match how hard the major feels in practice.

Overall, these metrics are useful for analyzing the structure of a curriculum and identifying potential risks, such as bottlenecks or delays. However, they do not capture everything about the student experience, so they should be used alongside real-world understanding of the courses.

In [5]:
using CurricularAnalytics
using CurricularVisualization
using CSV
using DataFrames
curric = read_csv("../data/Univ_of_Arizona-Aero.csv"); 
##this is just an example dataset, after running the script you can change this file path to the desired major

b = blocking_factor(curric)[2]
d = delay_factor(curric)[2]
c = centrality(curric)[2]

function top_k_courses(values, curric)
    
    top3 = sortperm(values, rev=true)[1:3] 
    
    return [curric.courses[i].name for i in top3]
end

top_blocking = top_k_courses(b, curric)
top_delay    = top_k_courses(d, curric)
top_central  = top_k_courses(c, curric)

println("Top 3 Blocking (take early):\n", top_blocking, "\n")
println("Top 3 Delay (don’t postpone):\n", top_delay, "\n")
println("Top 3 Centrality (connectors):\n", top_central, "\n")
println("Overall complexity: ", complexity(curric)[1])

Top 3 Blocking (take early):
["Calculus I w/ Applications", "Calculus II", "Intro Mechanics"]

Top 3 Delay (don’t postpone):
["Calculus I w/ Applications", "Calculus II", "Intro Mechanics"]

Top 3 Centrality (connectors):
["Calculus II", "Intro Mechanics", "Gasdynamics"]

Overall complexity: 460.0


## Test CSV reading

In [ ]:
df = CSV.read("../outputs/Actuarial_Science_BS_MTH_103.csv", DataFrame)

## test visualization

In [ ]:
curric = read_csv("../outputs/fake_data_sci.csv")
visualize(curric);